# ChakraModel V5 Strict Verification - True Anti-Fabrication Harness

This notebook copies your chosen dataset subset to the working directory, plants the Anti-Fabrication canaries, and runs the hardened `verify_kaggle.py` script via `harness_core`.

**Before running:**
1. Make sure you have added the `ChakraModel_Kaggle_Code` and `Anti-Fabrication Toolkit` datasets.
2. Ensure you run this using **Save Version -> Save & Run All (Commit)**.

In [12]:
!pip install -q ultralytics timm opencv-python-headless pycocotools

In [13]:
import os
import sys
import shutil
import glob
import json
import re

###################################################################
# CONFIGURATION
###################################################################
DATASET_ROOT = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'
VERIFY_DATASET_NAME = "PolypDB_BKAI"
WORKING_DATASET = '/kaggle/working/dataset_copy'

# 1. Clean up old working directory
if os.path.exists(WORKING_DATASET):
    shutil.rmtree(WORKING_DATASET)

# 2. Create the flat structure that the Harness strictly requires
os.makedirs(os.path.join(WORKING_DATASET, 'images'))
os.makedirs(os.path.join(WORKING_DATASET, 'masks'))

print(f"Flattening and copying files from all modalities in {DATASET_ROOT} ...")
image_count = 0
mask_count = 0

# 3. Recursively hunt down every image and mask in FICE, BLI, WLI, LCI
for root, dirs, files in os.walk(DATASET_ROOT):
    for file in files:
        # Ignore random hidden files
        if file.startswith('.') or not file.endswith(('.png', '.jpg', '.tif', '.bmp')):
            continue
            
        src = os.path.join(root, file)
        
        # We prefix the filename with the modality name (e.g. FICE_image1.png) 
        # so files with the same name from different modalities don't overwrite each other!
        modality = os.path.basename(os.path.dirname(root)) 
        new_filename = f"{modality}_{file}"
        
        if 'images' in root.lower():
            dst = os.path.join(WORKING_DATASET, 'images', new_filename)
            shutil.copy2(src, dst)
            image_count += 1
        elif 'masks' in root.lower():
            dst = os.path.join(WORKING_DATASET, 'masks', new_filename)
            shutil.copy2(src, dst)
            mask_count += 1

print(f"✅ Flattening Complete!")
print(f"Copied {image_count} real images and {mask_count} real masks into {WORKING_DATASET}")

Flattening and copying files from all modalities in /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI ...
✅ Flattening Complete!
Copied 1200 real images and 1200 real masks into /kaggle/working/dataset_copy


In [14]:
import os

def print_directory_tree(startpath, max_depth=4):
    print(f"🗺️ Directory tree for: {startpath}")
    print("=" * 50)
    
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        
        # Stop digging if we go too deep (keeps the output readable)
        if level > max_depth:
            continue
            
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root)
        if folder_name == '':
            folder_name = startpath
            
        print(f"{indent}📂 {folder_name}/")
        
        subindent = ' ' * 4 * (level + 1)
        
        # Print the first 3 files in this folder so we can see what type of data is here
        for f in files[:3]:
            print(f"{subindent}📄 {f}")
            
        if len(files) > 3:
            print(f"{subindent}... and {len(files)-3} more files")

print_directory_tree('/kaggle/input', max_depth=4)

🗺️ Directory tree for: /kaggle/input
📂 input/
    📂 datasets/
        📂 gokulraj324/
            📂 anti-fabrication-toolkit/
                📄 harness_core.py
        📂 gokulrocky/
            📂 chakramodel-kaggle-code/
                📄 requirements.txt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 train_pranet.py
                    ... and 31 more files
            📂 polypdb-polyp-raw/
                📂 PolypDB/
            📂 final-om-evlautation-upload/
                📄 Kaggle_ChakraTransformer_Evaluation.ipynb
                📂 weights/
                    📄 chakra_transformer_best.pth
                    📄 best.pt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 metrics_engine_v2.py
                    ... and 32 more files


In [15]:
%%writefile /kaggle/working/verify_kaggle.py
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path
import hashlib

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def verify_strict():
    print("=" * 60)
    print("STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION")
    print("=" * 60)
    
    # Identify codebase path
    code_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files and 'src' in root:
            code_dir = root
            break
    if not code_dir:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'chakranet_segmenter.py' in files:
                code_dir = root
                break
    assert code_dir, "FATAL: ChakraModel codebase not found in /kaggle/input"
    sys.path.insert(0, code_dir)
    print(f"[VERIFY] Codebase found at: {code_dir}")
    
    from ultralytics import YOLO
    from chakranet_segmenter import ChakraNet
    from metrics_engine_v2 import MetricsEngineV2

    # BUG 3 FIX: Strict Weight Selection
    yolo_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
    vit_candidates = glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)
    
    # Strictly filter yolo_candidates (ignore anything with 'transformer' in the path)
    yolo_candidates = [p for p in yolo_candidates if 'transformer' not in p.lower()]
    
    print(f"YOLO candidates found: {yolo_candidates}")
    print(f"ViT candidates found: {vit_candidates}")
    
    assert len(yolo_candidates) == 1, f"FATAL: Expected exactly 1 YOLO weight best.pt, found {len(yolo_candidates)}!"
    assert len(vit_candidates) == 1, f"FATAL: Expected exactly 1 ViT weight chakra_transformer_best.pth, found {len(vit_candidates)}!"
    
    yolo_path = yolo_candidates[0]
    weight_path = vit_candidates[0]

    print(f"Segmenter Checkpoint: {weight_path}")
    print(f"Segmenter MD5: {md5(weight_path)}")
    print(f"YOLO Checkpoint: {yolo_path}")
    print(f"YOLO MD5: {md5(yolo_path)}")
    print("=" * 60)
    
    nonce = os.environ.get("VERIFY_NONCE", "NO_NONCE")
    print(f"VERIFY_NONCE: {nonce}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing on device: {device}")
    
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)
    
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    
    # Load weights correctly by explicitly stripping 'module.'
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    res = segmenter.model.load_state_dict(sd_fixed, strict=False)
    print(f"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}")
    
    total_keys = len(segmenter.model.state_dict())
    if len(res.missing_keys) > (0.1 * total_keys):
        print(f"[FATAL] Major weight mismatch! Missing keys > 10% ({len(res.missing_keys)} missing out of {total_keys}).")
        sys.exit(1)
    
    metrics_engine = MetricsEngineV2()
    
    dataset_name = os.environ.get("VERIFY_DATASET_NAME")
    dataset_root = os.environ.get("VERIFY_DATASET_ROOT")
    
    if not dataset_name or not dataset_root:
        print("[FATAL] VERIFY_DATASET_NAME or VERIFY_DATASET_ROOT environment variable not set.")
        sys.exit(1)
        
    print(f"\n============================================================")
    print(f"VERIFYING DATASET: {dataset_name}")
    print(f"============================================================")
    
    d_root = Path(dataset_root)
    images_dir = d_root / "images"
    masks_dir = d_root / "masks"
    
    if not images_dir.exists():
        print(f"[FATAL] Images directory not found: {images_dir}")
        sys.exit(1)
        
    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    print(f"Total images found: {len(image_paths)}")
    
    results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    import csv
    csv_file = "/kaggle/working/per_image_metrics_log.csv"
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"])
        
    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential_path = masks_dir / (img_path.stem + ext)
            if potential_path.exists():
                mask_path = potential_path
                break
                
        if not mask_path:
            print(f"[WARN] Mask not found for {img_path.name}")
            continue
            
        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        
        img_resized = cv2.resize(img_bgr, (448, 448))
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        
        # Soft prob map initialized to 0
        c_prob = np.zeros((448, 448), dtype=np.float32)
        
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            box = boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            
            w_box, h_box = x2 - x1, y2 - y1
            px, py = int(0.25 * w_box), int(0.25 * h_box)
            px1, py1 = max(0, x1 - px), max(0, y1 - py)
            px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))
            
            if px2 > px1 and py2 > py1:
                roi_crop = img_resized[py1:py2, px1:px2]
                
                # BUG 2 FIX: Direct manual inference EXACTLY matching chakranet_segmenter.py
                from utils.transforms import letterbox_pad, unletterbox
                img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
                img_resized_crop, meta = letterbox_pad(img_rgb, target_size=(384, 384))
                
                img_norm = img_resized_crop.astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                img_norm = (img_norm - mean) / std
                img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)
                
                try:
                    with torch.no_grad():
                        logits = segmenter.model(img_tensor)
                        prob = torch.sigmoid(logits)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print("[WARN] Switched to CPU due to CUDA OutOfMemory!")
                        img_tensor_cpu = img_tensor.cpu()
                        segmenter.model.to('cpu')
                        with torch.no_grad():
                            logits = segmenter.model(img_tensor_cpu)
                            prob = torch.sigmoid(logits)
                        segmenter.model.to(device)
                    else:
                        raise
                        
                prob_np = prob.squeeze().cpu().numpy()
                prob_resized = unletterbox(prob_np, meta)
                
                c_prob[py1:py2, px1:px2] = prob_resized
        
        # Explicit binarization ONLY for the ground truth here (network outputs soft prob)
        gt_bin = (gt_resized > 127).astype(np.uint8)
        
        # Pass continuous probability map (c_prob) directly to compute_all 
        # (compute_all handles thresholding for Dice/IoU internally)
        metrics = metrics_engine.compute_all(c_prob, gt_bin)
        
        for k in results.keys():
            results[k].append(metrics[k])
            
        # BUG 4 FIX: Checkpoint saving per image
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([img_path.name, metrics['Dice'], metrics['mIoU'], metrics['wF-measure'], metrics['S-measure'], metrics['E-measure'], metrics['MAE']])
            
        # IMPORTANT: Print per-image dice WITH FILENAME for canary detection
        dice_score = metrics['Dice']
        print(f"File: {img_path.name} | Dice: {dice_score:.4f}")
        
    print(f"\n[DONE] {dataset_name}")
    print(f"Total Evaluated Images: {len(results['Dice'])}")
    for k in results.keys():
        print(f"Final True Average {k}: {float(np.mean(results[k])):.4f}")
    print(f"============================================================")

if __name__ == "__main__":
    verify_strict()



Overwriting /kaggle/working/verify_kaggle.py


In [16]:
harness_path = glob.glob('/kaggle/input/**/harness_core.py', recursive=True)
if not harness_path:
    harness_path = glob.glob('/kaggle/working/**/harness_core.py', recursive=True)

if not harness_path:
    print("FATAL: harness_core.py not found! Did you attach the Anti-Fabrication Toolkit dataset?")
    sys.exit(1)

toolkit_dir = os.path.dirname(harness_path[0])
sys.path.insert(0, toolkit_dir)

from harness_core import plant_multi_canary, run_live, verify, sign_verdict

print("Planting canaries...")
secret = plant_multi_canary(WORKING_DATASET)

extra_env = {
    "VERIFY_NONCE": secret["nonce"],
    "VERIFY_DATASET_NAME": VERIFY_DATASET_NAME,
    "VERIFY_DATASET_ROOT": WORKING_DATASET
}

print(f"\nRunning verification for {VERIFY_DATASET_NAME}...")
stdout_text, arrivals, elapsed, retcode = run_live('/kaggle/working/verify_kaggle.py', extra_env, timeout_sec=10000)

print(stdout_text)

if retcode != 0:
    print(f"FATAL: verify_kaggle.py exited with code {retcode}")
else:
    # Parse claimed total from stdout
    match = re.search(r"Total Evaluated Images:\s*(\d+)", stdout_text)
    claimed_total = int(match.group(1)) if match else -1
    
    # Use harness core to verify output logic
    real_count = len(os.listdir(os.path.join(WORKING_DATASET, "images")))
    verdict_dict = verify(stdout_text, arrivals, elapsed, secret, claimed_total, real_count)
    
    card = {
        "project": "ChakraModel",
        "dataset": VERIFY_DATASET_NAME,
        "nonce": secret["nonce"],
        "decision": "ADMITTED" if verdict_dict["pass"] else "WITHHELD",
        "findings": verdict_dict["findings"],
        "elapsed_sec": verdict_dict["elapsed_sec"]
    }
    
    signed_card = sign_verdict(card, secret["hmac_key"])
    
    with open('/kaggle/working/completion_card.json', 'w') as f:
        json.dump(signed_card, f, indent=2)
        
    print(f"\n=========================================")
    print(f"VERDICT: {card['decision']}")
    for finding in card['findings']:
        print(f"  - {finding}")
    print(f"Saved to completion_card.json")
    print(f"=========================================")


Planting canaries...

Running verification for PolypDB_BKAI...
STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION
[VERIFY] Codebase found at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
[WARN] HardwareMonitor unavailable: [Errno 30] Read-only file system: '/kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/logs'
Traceback (most recent call last):
  File "/kaggle/working/verify_kaggle.py", line 210, in <module>
    verify_strict()
  File "/kaggle/working/verify_kaggle.py", line 39, in verify_strict
    from metrics_engine_v2 import MetricsEngineV2
ModuleNotFoundError: No module named 'metrics_engine_v2'

FATAL: verify_kaggle.py exited with code 1


In [17]:
import os
bkai_path = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'

print(f"Contents of BKAI:")
items = os.listdir(bkai_path)
for item in items[:20]:
    path = os.path.join(bkai_path, item)
    if os.path.isdir(path):
        print(f"📂 [FOLDER] {item}")
    else:
        print(f"📄 [FILE]   {item}")
        
if len(items) > 20:
    print(f"... and {len(items)-20} more items.")

Contents of BKAI:
📂 [FOLDER] FICE
📂 [FOLDER] BLI
📂 [FOLDER] WLI
📂 [FOLDER] LCI


In [18]:
!pip install ultralytics

In [19]:
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path

# ---------------------------------------------------------------------------
# Locate ChakraModel codebase
# ---------------------------------------------------------------------------
# Add all src directories in Kaggle datasets to sys.path to ensure we find everything
src_dirs_added = []
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'src':
        sys.path.insert(0, root)
        src_dirs_added.append(root)

if not src_dirs_added:
    # Fallback just in case
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files:
            sys.path.insert(0, root)
            src_dirs_added.append(root)

print(f"[REPRODUCIBILITY] Added {len(src_dirs_added)} codebase directories to sys.path:")
for d in src_dirs_added:
    print(f"  -> {d}")

from ultralytics import YOLO
from chakranet_segmenter import ChakraNet
from metrics_engine_v2 import MetricsEngineV2


def compute_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    iou = intersection_area / float(box1_area + box2_area - intersection_area + 1e-6)
    return iou


def get_gt_bbox(mask_img):
    coords = cv2.findNonZero(mask_img)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    return [x, y, x + w, y + h]


def run_diagnostics():
    print("=" * 60)
    print("STARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)")
    print("=" * 60)

    # -----------------------------------------------------------------------
    # Locate weights - FIX #4: print exactly what's chosen, don't pick silently
    # -----------------------------------------------------------------------
    yolo_candidates = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
    vit_candidates = sorted(glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True))

    if not yolo_candidates or not vit_candidates:
        print("FATAL: could not find YOLO best.pt or chakra_transformer_best.pth")
        sys.exit(1)

    if len(yolo_candidates) > 1:
        print(f"[WARNING] Multiple YOLO checkpoints found: {yolo_candidates}")
    if len(vit_candidates) > 1:
        print(f"[WARNING] Multiple ChakraNet checkpoints found: {vit_candidates}")

    yolo_path = yolo_candidates[-1]
    weight_path = vit_candidates[-1]
    print(f"[REPRODUCIBILITY] YOLO checkpoint : {yolo_path}")
    print(f"[REPRODUCIBILITY] ChakraNet ckpt  : {weight_path}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Device: {device}")

    # Load YOLO
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)

    # -----------------------------------------------------------------------
    # Load ChakraNet - FIX #2: strict loading with explicit key reporting
    # -----------------------------------------------------------------------
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}

    load_result = segmenter.model.load_state_dict(sd_fixed, strict=False)
    if load_result.missing_keys:
        print(f"[CHECKPOINT WARNING] {len(load_result.missing_keys)} missing keys:")
        for k in load_result.missing_keys[:20]:
            print(f"    MISSING: {k}")
        if len(load_result.missing_keys) > 20:
            print(f"    ... and {len(load_result.missing_keys) - 20} more")
    if load_result.unexpected_keys:
        print(f"[CHECKPOINT WARNING] {len(load_result.unexpected_keys)} unexpected keys:")
        for k in load_result.unexpected_keys[:20]:
            print(f"    UNEXPECTED: {k}")
        if len(load_result.unexpected_keys) > 20:
            print(f"    ... and {len(load_result.unexpected_keys) - 20} more")
    if not load_result.missing_keys and not load_result.unexpected_keys:
        print("[CHECKPOINT OK] All keys matched exactly.")
    else:
        print("[CHECKPOINT WARNING] Mismatches above mean results may not reflect "
              "your intended trained weights. Investigate before trusting metrics below.")

    # FIX #1: eval mode - THIS WAS MISSING IN THE ORIGINAL DRAFT
    segmenter.model.eval()
    print("[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.")

    metrics_engine = MetricsEngineV2()

    images_dir = Path("/kaggle/working/dataset_copy/images")
    masks_dir = Path("/kaggle/working/dataset_copy/masks")

    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    image_paths = [p for p in image_paths if not p.name.startswith("CANARY")]

    print(f"Total valid diagnostic images: {len(image_paths)}")
    print("[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match "
          "your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline "
          "uses a different padding ratio, the oracle number below is not apples-to-apples "
          "with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.")

    yolo_ious = []
    yolo_hits_any = 0       # IoU > 0.0
    yolo_hits_50 = 0        # IoU >= 0.5  (FIX #3)
    yolo_misses = 0

    chakra_results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    per_image_dice = []     # FIX #5

    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential = masks_dir / (img_path.stem + ext)
            if potential.exists():
                mask_path = potential
                break

        if not mask_path:
            continue

        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        img_resized = cv2.resize(img_bgr, (448, 448))
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        gt_bin = (gt_resized > 127).astype(np.uint8)

        gt_bbox = get_gt_bbox(gt_bin)

        if gt_bbox is None:
            continue

        # ---------------------------------------------------------------
        # 1. DIAGNOSTIC: YOLO ONLY (Detection)
        # ---------------------------------------------------------------
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            pred_box = boxes[0].xyxy[0].cpu().numpy().tolist()

            iou = compute_iou(gt_bbox, pred_box)
            yolo_ious.append(iou)
            if iou > 0.0:
                yolo_hits_any += 1
            else:
                yolo_misses += 1
            if iou >= 0.5:
                yolo_hits_50 += 1
        else:
            yolo_ious.append(0.0)
            yolo_misses += 1

        # ---------------------------------------------------------------
        # 2. DIAGNOSTIC: CHAKRANET ONLY (Perfect Crops)
        # ---------------------------------------------------------------
        x1, y1, x2, y2 = gt_bbox
        w_box, h_box = x2 - x1, y2 - y1

        px, py = int(0.25 * w_box), int(0.25 * h_box)
        px1, py1 = max(0, x1 - px), max(0, y1 - py)
        px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))

        if px2 > px1 and py2 > py1:
            roi_crop = img_resized[py1:py2, px1:px2]

            img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
            img_resized_crop = cv2.resize(img_rgb, (384, 384))
            img_norm = img_resized_crop.astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_norm = (img_norm - mean) / std
            img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = segmenter.model(img_tensor)
                prob = torch.sigmoid(logits)

            prob_np = prob.squeeze().float().cpu().numpy().astype(np.float32)
            prob_resized = cv2.resize(prob_np, (px2 - px1, py2 - py1), interpolation=cv2.INTER_LINEAR)

            c_prob = np.zeros((448, 448), dtype=np.float32)
            c_prob[py1:py2, px1:px2] = prob_resized

            metrics = metrics_engine.compute_all(c_prob, gt_bin)
            for k in chakra_results.keys():
                chakra_results[k].append(metrics[k])
            per_image_dice.append(metrics["Dice"])

        if i % 100 == 0:
            print(f"Processed {i}/{len(image_paths)} images...")

    print("\n=========================================")
    print("YOLO DETECTION PERFORMANCE (DIAGNOSTIC)")
    print("=========================================")
    print(f"Average Bounding Box IoU: {np.mean(yolo_ious):.4f}")
    total_yolo = yolo_hits_any + yolo_misses
    print(f"Detection Hit Rate (IoU > 0.0):  {yolo_hits_any}/{total_yolo} ({yolo_hits_any/total_yolo*100:.2f}%)")
    print(f"Detection Recall  (IoU >= 0.5):  {yolo_hits_50}/{total_yolo} ({yolo_hits_50/total_yolo*100:.2f}%)")

    print("\n=========================================")
    print("CHAKRANET SEGMENTATION PERFORMANCE (PERFECT CROP)")
    print("=========================================")
    for k in chakra_results.keys():
        print(f"Perfect Crop Average {k}: {np.mean(chakra_results[k]):.4f}")

    if per_image_dice:
        arr = np.array(per_image_dice)
        print("\n=========================================")
        print("PER-IMAGE DICE DISTRIBUTION (bimodal check)")
        print("=========================================")
        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
        labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
        counts, _ = np.histogram(arr, bins=bins)
        for lbl, c in zip(labels, counts):
            print(f"  Dice {lbl}: {c} images ({c/len(arr)*100:.1f}%)")
        print(f"  Median Dice: {np.median(arr):.4f}  |  Std Dice: {np.std(arr):.4f}")
        print("  -> If 0.0-0.2 bucket is large relative to a strong 0.8-1.0 peak, "
              "that's a bimodal signal -> likely a DETECTION/localization problem "
              "(some images fail hard), not smooth segmentation weakness.")
    print("=========================================")


if __name__ == "__main__":
    run_diagnostics()


[REPRODUCIBILITY] Added 2 codebase directories to sys.path:
  -> /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
  -> /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/src
STARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)
[REPRODUCIBILITY] YOLO checkpoint : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/best.pt
[REPRODUCIBILITY] ChakraNet ckpt  : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth
[INFO] Device: cuda
2026-09-06 12:32:38,075 [HW-MONITOR] INFO Loading pretrained weights from Hugging Face hub (timm/vit_large_patch16_384.augreg_in21k_ft_in1k)
2026-09-06 12:32:38,516 [HW-MONITOR] INFO HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-09-06 12:32:38,517 [HW-MONITOR] WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-06 12:32:38,755 [HW-MONITOR] INFO HTTP Request: GET https://huggingface.co/api/models/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/xet-read-token/1a0edc93ef27935b955ce67a344087b140f85d47 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

2026-09-06 12:32:49,501 [HW-MONITOR] INFO [timm/vit_large_patch16_384.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
[WARN] ChakraNet: No weights found. Running with random initialization.
[CHECKPOINT OK] All keys matched exactly.
[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.
Total valid diagnostic images: 1200
[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline uses a different padding ratio, the oracle number below is not apples-to-apples with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4086: error: (-215:Assertion failed) func != 0 in function 'resize'
